# Laboratorio 1 - Regresión Lineal

### Imports

In [743]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler



from importlib.metadata import version
print(f"Versión de Pandas: {version('pandas')}")
print(f"Versión de Matplotlib: {version('matplotlib')}")
print(f"Versión de Seaborn: {version('seaborn')}")
print(f"Versión de numpy: {version('numpy')}")


Versión de Pandas: 2.2.3
Versión de Matplotlib: 3.10.0
Versión de Seaborn: 0.13.2
Versión de numpy: 2.3.4


### Carga de Archivos

In [744]:
datos_pacientes = pd.read_csv('./data/Datos Lab 1.csv', na_values=['NA', 'NaN', '', "null", 'None', 'Missing Value'])
datos = datos_pacientes.copy()
datos.head()

,Patient ID,Date of Service,Sex,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Blood Pressure (mmHg),Total Cholesterol (mg/dL),...,Physical Activity Level,Family History of CVD,Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Blood Pressure Category,Estimated LDL (mg/dL),CVD Risk Score,CVD Risk Level
0,isDx5313,"November 08, 2023",M,44.0,114.300,1.720,38.600,100.000,112/83,228.0,...,High,N,172.000,0.581,112.0,83.0,Hypertension Stage 1,121.0,19.880,HIGH
1,LHCK2961,20/03/2024,F,57.0,92.923,1.842,33.116,106.315,101/91,158.0,...,High,Y,184.172,0.577,101.0,91.0,Hypertension Stage 2,57.0,16.833,INTERMEDIARY
2,WjVn1699,2021-05-27,F,NaN,73.400,1.650,27.000,78.100,90/74,135.0,...,High,N,165.000,0.473,90.0,74.0,Normal,45.0,12.600,LOW
3,dCDO1109,"April 18, 2022",F,35.0,113.300,1.780,35.800,79.600,92/89,158.0,...,Moderate,Y,178.000,0.447,92.0,89.0,Hypertension Stage 1,94.0,14.920,HIGH
4,pnpE1080,01/11/2024,F,48.0,102.200,1.750,33.400,106.700,121/68,207.0,...,Low,Y,175.000,0.610,121.0,68.0,Elevated,128.0,18.870,HIGH


## 1. Exploración y Perfilamiento de los Datos

In [745]:
display(datos.sample(5))

,Patient ID,Date of Service,Sex,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Blood Pressure (mmHg),Total Cholesterol (mg/dL),...,Physical Activity Level,Family History of CVD,Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Blood Pressure Category,Estimated LDL (mg/dL),CVD Risk Score,CVD Risk Level
364,VdlB5483,03 Oct 23,M,55.0,85.000,1.750,27.800,79.400,94/71,240.0,...,Low,Y,175.000,0.454,94.0,71.0,Normal,178.0,17.060,HIGH
1415,GPCF6283,2023-11-19,M,48.0,71.900,1.740,23.700,90.900,127/71,116.0,...,Low,Y,174.000,0.522,127.0,71.0,Elevated,42.0,15.410,HIGH
533,NIDP9223,24 Nov 20,M,NaN,93.900,1.720,31.700,82.100,104/60,144.0,...,High,Y,172.000,0.477,104.0,60.0,Normal,64.0,14.420,INTERMEDIARY
549,srgw5250,23/11/2025,M,31.0,104.344,1.955,27.115,74.354,174/119,160.0,...,Low,N,195.475,0.380,174.0,NaN,Hypertension Stage 2,91.0,17.323,INTERMEDIARY
1084,GDej6322,26 Sep 22,M,55.0,88.834,1.665,27.856,91.370,120/75,106.0,...,Low,N,210.624,NaN,120.0,75.0,Elevated,39.0,13.691,INTERMEDIARY


Hay varios formatos de los datos en la columna Date of Service.

In [746]:
datos.shape

(1639, 24)

In [747]:
display(datos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1639 entries, 0 to 1638
Data columns (total 24 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Patient ID                    1639 non-null   object 
 1   Date of Service               1639 non-null   object 
 2   Sex                           1639 non-null   object 
 3   Age                           1571 non-null   float64
 4   Weight (kg)                   1566 non-null   float64
 5   Height (m)                    1578 non-null   float64
 6   BMI                           1586 non-null   float64
 7   Abdominal Circumference (cm)  1578 non-null   float64
 8   Blood Pressure (mmHg)         1639 non-null   object 
 9   Total Cholesterol (mg/dL)     1571 non-null   float64
 10  HDL (mg/dL)                   1557 non-null   float64
 11  Fasting Blood Sugar (mg/dL)   1585 non-null   float64
 12  Smoking Status                1639 non-null   object 
 13  Dia

None

Hay 24 columnas con 1639 datos.

Las columnas con valores nulos son: Age,  Weight (kg), Height (m), BMI, Abdominal Circumference (cm), Total Cholesterol (mg/dL),  HDL (mg/dL), Fasting Blood Sugar (mg/dL),  Height (cm),  Waist-to-Height Ratio, Systolic BP, Diastolic BP, Estimated LDL (mg/dL) y CVD Risk Score

La columna con más valores nulos tiene el 94,8% de todos los datos

In [748]:
dict = pd.read_excel('./data/DiccPacientes.xlsx')
pd.set_option('display.max_colwidth', None)
dict

,Nombre Columna,Tipo de dato,Comentarios
0,Patient ID,String,Identificador del paciente
1,Date of Service,Date,Fecha de la atención
2,Sex,String,"Sexo (Femenino, Masculino)"
3,Age,Integer,Edad
4,Weight (kg),Float,Peso
5,Height (m),Float,Altura
6,BMI,Float,Índice de masa corporal
7,Abdominal Circumference (cm),Float,Circunferencia abdominal
8,Blood Pressure (mmHg),String,"Presión sanguínea, de la forma ""<Presión arterial sistólica>/<Presión arterial diastólica>"""
9,Total Cholesterol (mg/dL),Float,Colesterol total


Los tipos de datos del el diccionario y pandas están distintos, a excepeción de los floats

In [749]:
datos.describe()

,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Total Cholesterol (mg/dL),HDL (mg/dL),Fasting Blood Sugar (mg/dL),Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Estimated LDL (mg/dL),CVD Risk Score
count,1571.000000,1566.000000,1578.000000,1586.000000,1578.000000,1571.000000,1557.000000,1585.000000,1571.000000,1563.000000,1578.000000,1554.000000,1582.000000,1610.000000
mean,46.803186,85.666006,1.757439,28.424744,91.538861,199.043673,56.183558,117.836860,175.770082,0.522440,125.632637,82.887536,113.235896,18.227281
std,13.039479,21.712504,0.118012,7.309275,13.427985,59.388670,16.721702,32.379634,11.695880,0.085692,22.577463,15.503625,61.435291,10.767666
min,6.134000,13.261000,1.371000,4.317000,49.542000,-1.256000,0.008000,15.306000,136.498000,0.250000,49.914000,31.720000,-92.055000,-20.057000
25%,37.000000,67.100000,1.666500,22.600000,79.700000,150.000000,42.000000,92.000000,167.000000,0.453000,108.000000,71.000000,62.000000,15.150000
50%,46.000000,86.314000,1.760000,28.000000,91.200000,199.000000,56.000000,115.000000,176.000000,0.519000,125.000000,82.000000,112.000000,16.967000
75%,55.000000,104.801500,1.850000,33.963000,102.267250,250.000000,70.000000,139.000000,185.000000,0.582000,141.000000,93.000000,159.000000,18.900000
max,89.420000,158.523000,2.146000,53.028000,136.336000,385.679000,110.315000,219.667000,214.394000,0.804000,202.711000,134.066000,317.314000,114.980000


Se identificaron múltiples problemas de calidad en los datos. En particular, se observaron valores imposibles físicamente como colesterol total y LDL negativos, así como puntajes de riesgo cardiovascular negativos, los cuales no tienen interpretación clínica directa. A su vez, es extraño tener un rango de edades tan grande.

También se evidencian outliers extremos en variables como presión arterial, altura y colesterol, que podrían influir de forma desproporcionada en modelos de regresión lineal.

In [750]:
((datos.isnull().sum()/datos.shape[0])).sort_values(ascending=False)

Diastolic BP                    0.051861
HDL (mg/dL)                     0.050031
Waist-to-Height Ratio           0.046370
Weight (kg)                     0.044539
Height (cm)                     0.041489
Age                             0.041489
Total Cholesterol (mg/dL)       0.041489
Height (m)                      0.037218
Abdominal Circumference (cm)    0.037218
Systolic BP                     0.037218
Estimated LDL (mg/dL)           0.034777
Fasting Blood Sugar (mg/dL)     0.032947
BMI                             0.032337
CVD Risk Score                  0.017694
Patient ID                      0.000000
Blood Pressure Category         0.000000
Smoking Status                  0.000000
Family History of CVD           0.000000
Physical Activity Level         0.000000
Diabetes Status                 0.000000
Date of Service                 0.000000
Blood Pressure (mmHg)           0.000000
Sex                             0.000000
CVD Risk Level                  0.000000
dtype: float64

In [751]:
nulos = datos[datos['Diastolic BP'].isna()].copy()

print(f"Filas con Diastolic BP nula: {nulos.shape[0]}")
indices = datos.index[datos['Diastolic BP'].isna()]
datos[datos['Diastolic BP'].isna()].copy().reset_index(drop=True)

Filas con Diastolic BP nula: 85


,Patient ID,Date of Service,Sex,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Blood Pressure (mmHg),Total Cholesterol (mg/dL),...,Physical Activity Level,Family History of CVD,Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Blood Pressure Category,Estimated LDL (mg/dL),CVD Risk Score,CVD Risk Level
0,nepF9907,06/08/2020,M,74.0,51.858,1.608,33.294,71.58,99/78,124.0,...,Moderate,Y,160.768,0.445,99.0,NaN,Normal,34.0,16.089,LOW
1,AhYt1346,09-28-2020,M,41.0,71.300,1.730,23.800,107.90,139/61,253.0,...,Low,Y,173.000,NaN,139.0,NaN,Hypertension Stage 1,146.0,16.770,HIGH
2,fHqx3420,"July 09, 2024",M,55.0,70.200,1.720,23.700,70.00,121/68,300.0,...,High,Y,172.000,0.407,121.0,NaN,Elevated,193.0,17.064,HIGH
3,NGwd5164,2021-12-31,M,33.0,74.000,1.760,23.900,77.40,118/86,165.0,...,High,Y,176.000,0.440,118.0,NaN,Hypertension Stage 1,64.0,13.980,LOW
4,ICKt4668,2025-11-01,M,42.0,59.400,1.890,16.600,94.40,125/61,228.0,...,High,N,189.000,0.499,125.0,NaN,Elevated,127.0,32.120,INTERMEDIARY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,cUUw7860,06-16-2023,F,37.0,75.400,1.800,23.300,86.50,149/78,296.0,...,High,N,NaN,0.481,149.0,NaN,Hypertension Stage 2,209.0,20.030,INTERMEDIARY
81,gynJ3543,25/01/2020,M,30.0,88.030,1.987,22.225,107.69,174/82,204.0,...,Moderate,Y,198.666,NaN,174.0,NaN,Hypertension Stage 1,124.0,19.225,HIGH
82,RsUs2064,2021-07-14,M,39.0,52.641,1.898,33.100,73.69,120/78,228.0,...,Moderate,Y,189.817,0.388,120.0,NaN,Elevated,160.0,19.180,INTERMEDIARY
83,ePpS4710,12/04/2022,M,48.0,50.100,1.770,16.000,104.10,146/95,210.0,...,Low,Y,177.000,0.588,146.0,NaN,Hypertension Stage 2,110.0,NaN,HIGH


In [752]:
nulos = datos[datos['Height (m)'].isna()].copy()

print(f"Filas con Height (m) nula: {nulos.shape[0]}")
indices = datos.index[datos['Height (m)'].isna()]
datos[datos['Height (m)'].isna()].copy().reset_index(drop=True)

Filas con Height (m) nula: 61


,Patient ID,Date of Service,Sex,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Blood Pressure (mmHg),Total Cholesterol (mg/dL),...,Physical Activity Level,Family History of CVD,Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Blood Pressure Category,Estimated LDL (mg/dL),CVD Risk Score,CVD Risk Level
0,oJnb5472,05 Jun 25,M,34.0,102.600,NaN,NaN,96.900,137/81,240.0,...,High,N,139.265,0.587,137.0,81.0,Hypertension Stage 1,133.0,19.190,INTERMEDIARY
1,NBUl2902,01-21-2025,F,62.0,87.544,NaN,38.527,86.487,154/60,125.0,...,Moderate,Y,178.337,0.485,154.0,60.0,Hypertension Stage 2,47.0,19.905,HIGH
2,MmNI0458,"May 31, 2024",M,48.0,NaN,NaN,28.800,79.700,102/63,245.0,...,Low,Y,178.000,0.448,102.0,63.0,Normal,143.0,15.760,HIGH
3,XZCU4466,22/01/2024,F,52.0,97.800,NaN,31.600,94.800,124/92,216.0,...,High,N,176.000,0.539,124.0,92.0,Hypertension Stage 2,136.0,18.840,HIGH
4,SdtW2617,21/06/2021,M,NaN,115.400,NaN,36.800,91.900,142/64,NaN,...,High,N,177.000,0.519,142.0,64.0,Hypertension Stage 2,67.0,19.180,HIGH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,inTX4760,12-10-2020,F,42.0,69.100,NaN,19.600,NaN,92/95,263.0,...,Low,N,188.000,0.545,92.0,95.0,Hypertension Stage 2,NaN,13.780,INTERMEDIARY
57,KeUy4784,01-23-2023,F,31.0,98.900,NaN,30.200,77.900,129/86,174.0,...,Low,Y,181.000,NaN,129.0,86.0,Hypertension Stage 1,107.0,15.970,HIGH
58,FMVb4279,06/03/2022,M,30.0,91.400,NaN,30.900,96.300,106/83,126.0,...,Moderate,N,172.000,0.560,106.0,83.0,Hypertension Stage 1,64.0,114.143,INTERMEDIARY
59,Qrlb4412,2025-05-02,M,73.0,118.888,NaN,35.129,97.390,101/95,147.0,...,Moderate,N,179.435,0.543,101.0,95.0,Hypertension Stage 2,74.0,17.016,LOW


Existen datos que tienen la altura en centimetros pero no en metros y viceversa

In [753]:
datos.duplicated(keep = False).sum()

np.int64(302)

In [754]:
# ids duplicados
dup_counts = (datos['Patient ID'].value_counts()
                        .loc[lambda s: s > 1]
                        .sort_values(ascending=False))
for id_, n in dup_counts.items():
    print(f"Id={id_} → {n} apariciones")

Id=THSJ8564 → 3 apariciones
Id=LdMm6225 → 3 apariciones
Id=lqUc7918 → 3 apariciones
Id=ChGR7779 → 3 apariciones
Id=szyX1885 → 3 apariciones
Id=PHii0023 → 3 apariciones
Id=svjU9851 → 3 apariciones
Id=sINw8855 → 3 apariciones
Id=EsZI4603 → 3 apariciones
Id=zZle5455 → 3 apariciones
Id=KQUs0708 → 3 apariciones
Id=HNpy8362 → 3 apariciones
Id=tlWh1188 → 3 apariciones
Id=VgtQ3260 → 3 apariciones
Id=HNwp6592 → 3 apariciones
Id=gdBF9655 → 3 apariciones
Id=ivTX4645 → 3 apariciones
Id=Xvbe7861 → 3 apariciones
Id=gXxm9399 → 3 apariciones
Id=TPHC2670 → 3 apariciones
Id=INdI1482 → 3 apariciones
Id=nmgP2712 → 3 apariciones
Id=FTEC4446 → 3 apariciones
Id=rfYk3516 → 3 apariciones
Id=RlsB8509 → 3 apariciones
Id=dJuC5084 → 3 apariciones
Id=STpP5810 → 3 apariciones
Id=GhAN9460 → 3 apariciones
Id=pEpZ9034 → 3 apariciones
Id=dSiv4949 → 3 apariciones
Id=cUUw7860 → 3 apariciones
Id=UIWC3599 → 3 apariciones
Id=ZDBx7052 → 3 apariciones
Id=fHqx3420 → 3 apariciones
Id=YLCe2926 → 3 apariciones
Id=kajW6905 → 3 apar

In [755]:
# Detalle de los registros
dup_table = (
    datos[datos['Patient ID'].duplicated(keep=False)]
    .copy()
    .assign(repeticiones=datos.groupby('Patient ID')['Patient ID'].transform('size'))
    .sort_values(['repeticiones', 'Patient ID'], ascending=[False, True])
)

# Mostrar tabla
dup_table

,Patient ID,Date of Service,Sex,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Blood Pressure (mmHg),Total Cholesterol (mg/dL),...,Family History of CVD,Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Blood Pressure Category,Estimated LDL (mg/dL),CVD Risk Score,CVD Risk Level,repeticiones
17,AhYt1346,09-28-2020,M,41.0,71.300,1.730,23.800,107.9,139/61,253.0,...,Y,173.000,NaN,139.0,NaN,Hypertension Stage 1,146.0,16.770,HIGH,3
1227,AhYt1346,09-28-2020,M,41.0,71.300,1.730,23.800,107.9,139/61,253.0,...,Y,173.000,NaN,139.0,NaN,Hypertension Stage 1,146.0,-13.090,HIGH,3
1584,AhYt1346,09-28-2020,M,41.0,71.300,1.730,23.800,107.9,139/61,253.0,...,Y,173.000,NaN,139.0,NaN,Hypertension Stage 1,146.0,16.770,HIGH,3
130,BQvQ6431,09/11/2020,M,33.0,118.300,1.690,41.400,72.1,116/93,171.0,...,N,210.554,0.427,116.0,93.0,Hypertension Stage 2,97.0,17.500,LOW,3
1469,BQvQ6431,09/11/2020,M,33.0,118.300,1.690,41.400,72.1,116/93,171.0,...,N,210.554,0.427,116.0,93.0,Hypertension Stage 2,97.0,29.833,LOW,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
915,yvsn3005,28 Oct 20,F,60.0,54.300,1.810,16.600,99.1,133/65,187.0,...,N,181.000,0.548,133.0,65.0,Hypertension Stage 1,79.0,15.710,INTERMEDIARY,2
95,zcgB3048,12/06/2020,M,38.0,60.436,1.506,20.269,79.7,168/63,119.0,...,N,150.616,0.529,168.0,63.0,Hypertension Stage 2,43.0,14.834,HIGH,2
375,zcgB3048,12/06/2020,M,38.0,60.436,1.506,20.269,79.7,168/63,119.0,...,N,150.616,0.529,168.0,63.0,Hypertension Stage 2,43.0,14.834,HIGH,2
799,zxhX5525,"November 13, 2021",M,26.0,58.953,1.688,25.286,NaN,110/114,258.0,...,Y,168.763,0.455,110.0,114.0,Hypertension Stage 2,170.0,15.717,HIGH,2


In [756]:
datos['Date of Service'].value_counts()

Date of Service
09-20-2023           6
December 05, 2025    5
June 08, 2023        4
December 02, 2020    4
05-30-2025           4
                    ..
2024-09-24           1
11/09/2020           1
August 25, 2024      1
31/07/2022           1
2021-05-01           1
Name: count, Length: 1274, dtype: int64

In [757]:
profile = ProfileReport(
    datos,
    title="Reporte de Análisis Exploratorio",
    explorative=True
)

In [ ]:
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 24/24 [00:00<00:00, 34.80it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

### Hallazgos clave de la exploración
- Se identifican valores nulos principalmente en Distolic BP, HDL (mg/dL), Age, Weight, Height y BMI.
- Existen varios formatos para las fechas.
- Hay varios registros de pacientes repetidos.
- Los tipos de datos entre el Pandas y el diccionario no concuerdan
- Hay datos que si bien hacen sentido matemático, no se atienen a la lógica del negocio


## 2. Limpieza y Preparación

#### Corregir inconsistencia de alturas 

In [ ]:
# Calcular Height (m) con BMI y Weight (kg) donde Height (m) es nulo
mask_height_nan = datos['Height (m)'].isna() & datos['Weight (kg)'].notna() & datos['BMI'].notna()
datos.loc[mask_height_nan, 'Height (m)'] = (datos.loc[mask_height_nan, 'Weight (kg)'] / datos.loc[mask_height_nan, 'BMI'])

# Caso 1: Height (m) vacío y Height (cm) disponible
mask_m_nan = datos['Height (m)'].isna() & datos['Height (cm)'].notna()
datos.loc[mask_m_nan, 'Height (m)'] = datos.loc[mask_m_nan, 'Height (cm)'] / 100

# Caso 2: Height (cm) vacío y Height (m) disponible
mask_cm_nan = datos['Height (cm)'].isna() & datos['Height (m)'].notna()
datos.loc[mask_cm_nan, 'Height (cm)'] = datos.loc[mask_cm_nan, 'Height (m)'] * 100

# Verificación rápida
datos[['Height (m)', 'Height (cm)']].isna().sum()


Height (m)     0
Height (cm)    0
dtype: int64

Se corrigió las inconsistencias entre las alturas en cm y m

#### Calcular Weight (kg) con BMI y Height (kg)

In [ ]:
mask_weight_nan = datos['Weight (kg)'].isna() & datos['BMI'].notna() & datos['Height (m)'].notna()
datos.loc[mask_weight_nan, 'Weight (kg)'] = datos.loc[mask_weight_nan, 'BMI'] * (datos.loc[mask_weight_nan, 'Height (m)'] ** 2)

print(f"Valores nulos en Weight (kg) después de la imputación: {datos['Weight (kg)'].isna().sum()}")

Valores nulos en Weight (kg) después de la imputación: 1


In [ ]:
mask_bmi_nan = datos['BMI'].isna() & datos['Weight (kg)'].notna() & datos['Height (m)'].notna()
before = datos['BMI'].isna().sum()
datos.loc[mask_bmi_nan, 'BMI'] = datos.loc[mask_bmi_nan, 'Weight (kg)'] / (datos.loc[mask_bmi_nan, 'Height (m)'] ** 2)
after = datos['BMI'].isna().sum()
print(f"BMI imputados: {before - after}")
print(f"BMI nulos restantes: {after}")

BMI imputados: 52
BMI nulos restantes: 1


In [ ]:
datos['Weight (kg)'].isna().sum()

np.int64(1)

In [ ]:
datos_borrar = datos[datos['BMI'].isna() & datos['Weight (kg)'].isna()]
datos = datos.drop(datos_borrar.index)

datos['BMI'].isna().sum() & datos['Weight (kg)'].isna().sum()

np.int64(0)

Se imputaron los datos faltantes de BMI, Height y Weight usando la formula de BMI = Weight / Height

#### Calcular Waist to Height ratio con Altura y Circunferencia abdominal

In [ ]:
mask_whr_nan = datos['Waist-to-Height Ratio'].isna() & datos['Abdominal Circumference (cm)'].notna() & datos['Height (cm)'].notna()
before = datos['Waist-to-Height Ratio'].isna().sum()
datos.loc[mask_whr_nan, 'Waist-to-Height Ratio'] = datos.loc[mask_whr_nan, 'Abdominal Circumference (cm)'] / datos.loc[mask_whr_nan, 'Height (cm)']
after = datos['Waist-to-Height Ratio'].isna().sum()
print(f"Waist-to-Height Ratio imputados: {before - after}")
print(f"Waist-to-Height Ratio nulos restantes: {after}")

Waist-to-Height Ratio imputados: 75
Waist-to-Height Ratio nulos restantes: 1


In [ ]:
mask_ac_nan = datos['Abdominal Circumference (cm)'].isna() & datos['Waist-to-Height Ratio'].notna() & datos['Height (cm)'].notna()
before = datos['Abdominal Circumference (cm)'].isna().sum()
datos.loc[mask_ac_nan, 'Abdominal Circumference (cm)'] = datos.loc[mask_ac_nan, 'Waist-to-Height Ratio'] / datos.loc[mask_ac_nan, 'Height (cm)']
after = datos['Abdominal Circumference (cm)'].isna().sum()
print(f"Abdominal Circumference (cm) imputados: {before - after}")
print(f"Abdominal Circumference (cm) nulos restantes: {after}")

Abdominal Circumference (cm) imputados: 60
Abdominal Circumference (cm) nulos restantes: 1


Se imputaron los datos de Waist-to-Height Ratio usando la formula Ratio = Abdominal Circumference / Height (cm)

#### Columnas con valores negativos

In [ ]:
cols_to_impute = ['Estimated LDL (mg/dL)', 'Total Cholesterol (mg/dL)']

for col in cols_to_impute:
    negative_value = datos[datos[col] >= 0][col]*-1
    datos.loc[datos[col] < 0, col] = negative_value

print("Valores negativos imputados:")
for col in cols_to_impute:
    print(f"{col}: {(datos[col] < 0).sum()} valores negativos restantes")

Valores negativos imputados:
Estimated LDL (mg/dL): 0 valores negativos restantes
Total Cholesterol (mg/dL): 0 valores negativos restantes


Los valores que no tenian sentido para el negocio se cambiaron a positivos

#### Borrar datos faltantes de CVD risk Score

In [ ]:
# Convertir numpy NaN a pd.NA en la columna 'CVD Risk Score'
datos['CVD Risk Score'] = datos['CVD Risk Score'].replace({np.nan: pd.NA})
print(f"Nulos en 'CVD Risk Score': {datos['CVD Risk Score'].isna().sum()}")
datos = datos.dropna(subset=['CVD Risk Score'])
datos['CVD Risk Score'].isna().sum()

Nulos en 'CVD Risk Score': 29


np.int64(0)

Como CVD Risk Score se va a usar como target, entonces se eliminan los datos faltantes porque estariamos introduciendo un sesgo

In [ ]:
bp_split = datos['Blood Pressure (mmHg)'].str.split('/', expand=True)
display(datos.loc[:,['Blood Pressure (mmHg)', 'Systolic BP', 'Diastolic BP']])

datos['Blood Pressure (mmHg)'].dtype


,Blood Pressure (mmHg),Systolic BP,Diastolic BP
0,112/83,112.0,83.0
1,101/91,101.0,91.0
2,90/74,90.0,74.0
3,92/89,92.0,89.0
4,121/68,121.0,68.0
...,...,...,...
1633,145/92,145.0,92.0
1634,124/90,124.0,90.0
1635,95/111,95.0,111.0
1637,144/91,144.0,NaN


dtype('O')

#### Separar presión arterial y copiar los datos en Systolic y Distolic

In [ ]:
bp_split = datos['Blood Pressure (mmHg)'] \
    .str.replace(" ", "") \
    .str.split('/', expand=True)


datos['Systolic BP'] = pd.to_numeric(bp_split[0], errors='coerce')
datos['Diastolic BP'] = pd.to_numeric(bp_split[1], errors='coerce')

datos.drop(columns=['Blood Pressure (mmHg)'], inplace=True)


La variable Blood Pressure (mmHg) se encontraba almacenada como texto en formato
"sistólica/diastólica". Se realizó su separación en dos variables numéricas
para permitir su uso en modelos de regresión.

Posteriormente se eliminaron valores fisiológicamente imposibles e inconsistencias
lógicas (diastólica ≥ sistólica). Finalmente, se crearon variables derivadas
como presión de pulso y presión arterial media para enriquecer la información clínica.

#### Eliminar categoria de presion arterial

In [ ]:
datos = datos.drop(columns=['Blood Pressure Category'])

La variable Blood Pressure Category fue eliminada del modelo debido a que
se deriva directamente de las variables numéricas de presión arterial.
Incluir ambas introduciría redundancia y multicolinealidad,
afectando la estabilidad del modelo de regresión lineal.

In [ ]:
datos = datos.drop(columns=['CVD Risk Level'])

#### Imputar datos faltantes

In [ ]:
cols_to_impute = ['Age', 
                  'Estimated LDL (mg/dL)', 
                  'Fasting Blood Sugar (mg/dL)', 'Total Cholesterol (mg/dL)', 
                  'HDL (mg/dL)', 'Abdominal Circumference (cm)', 'Waist-to-Height Ratio']

for col in cols_to_impute:
    med = datos[col].mean()
    datos.loc[:, col] = datos.loc[:, col].fillna(med)
    print(f"{col}: {datos[col].isna().sum()} valores nulos restantes")

Age: 0 valores nulos restantes
Estimated LDL (mg/dL): 0 valores nulos restantes
Fasting Blood Sugar (mg/dL): 0 valores nulos restantes
Total Cholesterol (mg/dL): 0 valores nulos restantes
HDL (mg/dL): 0 valores nulos restantes
Abdominal Circumference (cm): 0 valores nulos restantes
Waist-to-Height Ratio: 0 valores nulos restantes


Terminamos de limpiar la completitud de los datos

#### Agrupar registros repetidos

In [ ]:
def first_non_null(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else pd.NA

agg = {}
for col in datos.columns:
    if col == 'Patient ID':
        continue
    if pd.api.types.is_numeric_dtype(datos[col]):
        agg[col] = 'mean'
    else:
        agg[col] = first_non_null

datos_unicos = datos.groupby('Patient ID', as_index=False).agg(agg)

print(f"Registros antes: {datos.shape[0]}, después: {datos_unicos.shape[0]}")
datos = datos_unicos

print(f"Registros duplicados: {datos['Patient ID'].duplicated().sum()}")
datos.info()

Agrupamos todos los pacientes repetidos

In [ ]:
# Interacciones entre variables importantes
datos['BMI_Age'] = datos['BMI'] * datos['Age']
datos['BMI_Systolic'] = datos['BMI'] * datos['Systolic BP']
datos['Age_Cholesterol'] = datos['Age'] * datos['Total Cholesterol (mg/dL)']
datos['Systolic_Diastolic'] = datos['Systolic BP'] * datos['Diastolic BP']

#  Transformaciones no lineales

# Logaritmo de variables con sesgo positivo (agregamos 1 para evitar log(0))
for col in ['Total Cholesterol (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'BMI']:
    if col in datos.columns:
        datos[f'{col}_log'] = np.log1p(datos[col])

# Cuadrados de variables importantes
for col in ['Age', 'BMI', 'Systolic BP']:
    if col in datos.columns:
        datos[f'{col}_squared'] = datos[col] ** 2


# Ratios y combinaciones

# Ratio Colesterol Total / HDL (indicador de riesgo cardiovascular)
datos['Cholesterol_HDL_Ratio'] = datos['Total Cholesterol (mg/dL)'] / (datos['HDL (mg/dL)'] + 1)

# Presión arterial media (MAP)
datos['Mean_Arterial_Pressure'] = (datos['Systolic BP'] + 2 * datos['Diastolic BP']) / 3

# Presión de pulso
datos['Pulse_Pressure'] = datos['Systolic BP'] - datos['Diastolic BP']

In [ ]:
datos['Atherogenic_Index'] = (datos['Total Cholesterol (mg/dL)'] - 
                                        datos['HDL (mg/dL)']) / datos['HDL (mg/dL)']
    
# Ratio Triglicéridos/HDL (indicador de resistencia a insulina)
# Usamos glucosa como proxy de triglicéridos si no están disponibles
datos['Glucose_HDL_Ratio'] = (datos['Fasting Blood Sugar (mg/dL)'] / 
                                        (datos['HDL (mg/dL)'] + 1))
    
# Índice de Masa Corporal ajustado por edad
datos['BMI_Age_Adjusted'] = datos['BMI'] * (datos['Age'] / 50)
    
# Presión de pulso ajustada por edad (marcador de rigidez arterial)
datos['Pulse_Pressure_Age_Adjusted'] = ((datos['Systolic BP'] - 
                                                   datos['Diastolic BP']) * 
                                                   datos['Age'] / 50)
    
# Ratio Peso/Altura cúbica (mejor que BMI para ciertos casos)
datos['Ponderal_Index'] = datos['Weight (kg)'] / (datos['Height (m)'] ** 3)
    
# Carga de presión (combina sistólica y diastólica)
datos['BP_Load'] = (datos['Systolic BP'] * 0.7 + 
                              datos['Diastolic BP'] * 0.3)
    
# Ratio Cintura/Altura al cuadrado (mejor predictor que WHR simple)
datos['WHR_Squared'] = datos['Waist-to-Height Ratio'] ** 2
    
# Índice de riesgo lipídico
datos['Lipid_Risk_Index'] = (datos['Total Cholesterol (mg/dL)'] / 
                                       (datos['HDL (mg/dL)'] + 1)) * datos['BMI']
    

## 3. Modelos de regresión lineal

In [ ]:
numeric_features = datos.select_dtypes(include=[np.number]).columns.tolist()
numeric_features.remove('CVD Risk Score')

# Calcular correlaciones
correlations = datos[numeric_features + ['CVD Risk Score']].corr()['CVD Risk Score'].drop('CVD Risk Score')
correlations = correlations.sort_values(ascending=False)

ValueError: list.remove(x): x not in list

In [ ]:
# Separar características y variable objetivo
X = datos.drop('CVD Risk Score', axis=1)
y = datos['CVD Risk Score']

# División train/test según especificaciones
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25, 
    random_state=42
)

In [ ]:
# Identificar tipos de columnas
categorical_binary = ['Sex']  # Variables binarias (M/F)

categorical_ordinal = [
    'Physical Activity Level',  # Low < Moderate < High
]


# Variables numéricas (todas las que no son categóricas)
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Variables categóricas binarias convertidas a numéricas (0/1)
binary_features = [
    'Smoking Status',      # 0: No (N), 1: Yes (Y)
    'Diabetes Status',     # 0: No (N), 1: Yes (Y)  
    'Family History of CVD' # 0: No (N), 1: Yes (Y)
]

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder


preprocessor_1 = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        
        ('bin_manual', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(categories=[['N', 'Y']] * len(binary_features)))
        ]), binary_features),
        
        ('cat_binary', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first', sparse_output=False))
        ]), categorical_binary),
        
        ('cat_ordinal', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(categories=[['Low', 'Moderate', 'High']]))
        ]), categorical_ordinal),
        
    ],
    remainder='drop'
)

In [ ]:
# Seleccionar top features por correlación
correlations_abs = correlations.abs().sort_values(ascending=False)
top_features = correlations_abs.head(20).index.tolist()

# Separar en numéricas y categóricas
top_numeric = [f for f in top_features if f in numeric_features]
top_binary = [f for f in top_features if f in binary_features]
top_cat_bin = [f for f in top_features if f in categorical_binary]
top_cat_ord = [f for f in top_features if f in categorical_ordinal]

preprocessor_2 = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), top_numeric),
        
        ('bin_manual', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(categories=[['N', 'Y']] * len(top_binary)))
        ]), top_binary) if len(top_binary) > 0 else ('bin_manual', 'drop', []),
        
        ('cat_binary', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first', sparse_output=False))
        ]), top_cat_bin) if len(top_cat_bin) > 0 else ('cat_binary', 'drop', []),
        
        ('cat_ordinal', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(categories=[['Low', 'Moderate', 'High']]))
        ]), top_cat_ord) if len(top_cat_ord) > 0 else ('cat_ordinal', 'drop', []),
    ],
    remainder='drop'
)

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Evalúa un modelo y retorna métricas"""
    # Predicciones
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calcular métricas
    metrics = {
        'Modelo': model_name,
        'RMSE_train': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'RMSE_test': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'MAE_train': mean_absolute_error(y_train, y_train_pred),
        'MAE_test': mean_absolute_error(y_test, y_test_pred),
        'R2_train': r2_score(y_train, y_train_pred),
        'R2_test': r2_score(y_test, y_test_pred)
    }
    
    return metrics, y_test_pred

def print_metrics(metrics):
    """Imprime métricas de forma formateada"""
    print(f"\n{'='*80}")
    print(f"{metrics['Modelo']}")
    print(f"{'='*80}")
    print(f"RMSE - Train: {metrics['RMSE_train']:.4f} | Test: {metrics['RMSE_test']:.4f}")
    print(f"MAE  - Train: {metrics['MAE_train']:.4f} | Test: {metrics['MAE_test']:.4f}")
    print(f"R²   - Train: {metrics['R2_train']:.4f} | Test: {metrics['R2_test']:.4f}")
    
    # Detectar overfitting
    if metrics['R2_train'] - metrics['R2_test'] > 0.1:
        print("\n Posible overfitting detectado (gran diferencia entre train y test)")
    elif metrics['R2_test'] < 0:
        print("\n Modelo con rendimiento muy pobre (R² negativo)")
    elif metrics['R2_test'] > 0.7:
        print("\n Excelente rendimiento del modelo")
    elif metrics['R2_test'] > 0.5:
        print("\n Buen rendimiento del modelo")
    else:
        print("\n Rendimiento moderado - considerar más ingeniería de features")


In [ ]:
model_1 = Pipeline([
    ('preprocessor', preprocessor_1),
    ('regressor', LinearRegression())
])

model_1.fit(X_train, y_train)
metrics_1, pred_1 = evaluate_model(model_1, X_train, X_test, y_train, y_test, 
                                   "Modelo 1: Linear Regression (Todas las features)")
print_metrics(metrics_1)


Modelo 1: Linear Regression (Todas las features)
RMSE - Train: 10.4279 | Test: 10.9451
MAE  - Train: 3.3089 | Test: 3.2989
R²   - Train: 0.0815 | Test: 0.0013

 Rendimiento moderado - considerar más ingeniería de features


In [ ]:
model_2 = Pipeline([
    ('preprocessor', preprocessor_2),
    ('regressor', LinearRegression())
])

model_2.fit(X_train, y_train)
metrics_2, pred_2 = evaluate_model(model_2, X_train, X_test, y_train, y_test,
                                   "Modelo 2: Linear Regression (Top Features)")
print_metrics(metrics_2)


Modelo 2: Linear Regression (Top Features)
RMSE - Train: 10.6090 | Test: 10.8889
MAE  - Train: 3.1633 | Test: 3.1640
R²   - Train: 0.0493 | Test: 0.0115

 Rendimiento moderado - considerar más ingeniería de features


In [ ]:
model_3 = Pipeline([
    ('preprocessor', preprocessor_1),
    ('regressor', Ridge(alpha=1.0, random_state=42))
])

model_3.fit(X_train, y_train)
metrics_3, pred_3 = evaluate_model(model_3, X_train, X_test, y_train, y_test,
                                   "Modelo 3: Ridge Regression (α=1.0)")
print_metrics(metrics_3)


Modelo 3: Ridge Regression (α=1.0)
RMSE - Train: 10.4305 | Test: 10.9324
MAE  - Train: 3.2800 | Test: 3.2660
R²   - Train: 0.0810 | Test: 0.0036

 Rendimiento moderado - considerar más ingeniería de features


In [ ]:
from sklearn.linear_model import Lasso


model_4 = Pipeline([
    ('preprocessor', preprocessor_1),
    ('regressor', Lasso(alpha=0.1, random_state=42, max_iter=10000))
])

model_4.fit(X_train, y_train)
metrics_4, pred_4 = evaluate_model(model_4, X_train, X_test, y_train, y_test,
                                   "Modelo 4: Lasso Regression (α=0.1)")
print_metrics(metrics_4)


Modelo 4: Lasso Regression (α=0.1)
RMSE - Train: 10.4963 | Test: 10.8100
MAE  - Train: 2.9940 | Test: 2.9754
R²   - Train: 0.0694 | Test: 0.0258

 Rendimiento moderado - considerar más ingeniería de features
